# 02b - Chunk and embed the filings (table-aware version)

Same steps as `02a`, with one difference: tables are not flattened into a run of numbers. Every value is written
together with its row and its column, for example `iPhone: 2026 = $254,098; Change 2026 vs 2025 = 21%`, and each
table becomes its own chunk (a long table is cut between rows, the title line is repeated in every piece).

The input is the raw HTML saved by notebook `01`, so the filings are parsed again here (no network needed).
Output goes to the folder `tables/` of the private dataset; the text version in `text/` stays untouched.

**Before running:** Runtime > T4 GPU, and the Colab Secrets `HF_TOKEN` (write access). About 10 minutes.

In [ ]:
import os
import subprocess
import sys

from google.colab import userdata

REPO = "10K-agentic-rag"
BRANCH = "main"
try:  # only needed while the repository is private (a read-only token for this one repository)
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001  the secret does not exist: the repository is public
    token = None
login = f"{token}@" if token else ""
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://{login}github.com/ilaria-cesa/{REPO}.git"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never hard-code the token
HF_REPO = "IlariaCesa/10k-agentic-rag-data"

Get the raw filings, parse them again with structured tables, and cut them into chunks (CPU, no model needed).
Same chunk size (1600 characters, overlap 200) as the text version, so the only difference is how tables are written.

In [ ]:
from src.chunking.run import main as make_chunks
from src.hub.dataset import download_folder
from src.ingest.reparse import main as reparse

download_folder(HF_REPO, "parsed", Path("data"))  # metadata of each filing (company, dates)
download_folder(HF_REPO, "raw", Path("data"))  # original HTML
reparse(["--out", "data", "--tables", "structured"])
make_chunks(["--out", "data", "--tables"])

How well were the tables read? Counts only: this does not say whether each value sits under the right column,
that was only checked by hand on a few filings.

In [ ]:
from collections import Counter

from src.retrieval.embed import load_chunks

chunks = load_chunks("data/chunks_tables_1600.jsonl")
table_chunks = [c for c in chunks if c["text"].startswith("Table")]
texts = [c["embed_text"] for c in chunks]
print(
    len(chunks), "chunks,", len(table_chunks), f"are tables ({len(table_chunks) / len(chunks):.1%})"
)
print("table chunks per section:", dict(Counter(c["section"] for c in table_chunks)))
print("table chunks per company:", dict(sorted(Counter(c["ticker"] for c in table_chunks).items())))
print("\nExample:\n" + table_chunks[len(table_chunks) // 2]["text"][:900])

Real token length of every chunk. Tables are full of numbers, which take more tokens than words, so a
1600-character table chunk can be longer than a text chunk. The limit is raised just enough that nothing is cut,
and the value used is saved in the report.

In [ ]:
from sentence_transformers import SentenceTransformer

from src.retrieval.embed import length_report, token_lengths

MODEL_NAME = "BAAI/bge-m3"
model = SentenceTransformer(MODEL_NAME, device="cuda", model_kwargs={"dtype": torch.float16})

lengths = token_lengths(texts, model.tokenizer)
MAX_LEN = max(640, -(-max(lengths) // 64) * 64)  # round up to a multiple of 64
assert MAX_LEN <= 1024, "table chunks are far too long in tokens: make them smaller in the chunker"
model.max_seq_length = MAX_LEN
report = length_report(lengths, MAX_LEN)
report["chars_per_token"] = round(sum(len(t) for t in texts) / sum(lengths), 2)
table_idx = [i for i, c in enumerate(chunks) if c["text"].startswith("Table")]
report["table_chunks_max_tokens"] = max(lengths[i] for i in table_idx)
report["table_chunks_chars_per_token"] = round(
    sum(len(texts[i]) for i in table_idx) / sum(lengths[i] for i in table_idx), 2
)
assert report["n_over_limit"] == 0
print(report)

In [ ]:
from src.retrieval.embed import embed_texts

embeddings = embed_texts(model, texts, batch_size=32)
print(embeddings.shape, embeddings.dtype)

Smoke test (not an evaluation): two numeric questions and one text question.

In [ ]:
from src.retrieval.index import build_index, search

index = build_index(embeddings)
for q in [
    "What were Apple's net sales in Greater China and how much did they change?",
    "What was JPMorgan's net interest income?",
    "How does Apple describe supply chain risks?",
]:
    scores, ids = search(index, embed_texts(model, [q], progress=False), k=3)
    print("\nQ:", q)
    for s, i in zip(scores[0], ids[0]):
        print(f"  {s:.3f}  {chunks[i]['chunk_id']}  {chunks[i]['text'][:110]!r}")

Save everything the next notebooks need in the `tables/` folder of the private dataset.

In [ ]:
import importlib.metadata as md
import json

from src.hub.dataset import upload_folder

out = Path("out_tables")
out.mkdir(exist_ok=True)
report["model"] = MODEL_NAME
report["versions"] = {
    p: md.version(p)
    for p in [
        "sentence-transformers",
        "transformers",
        "torch",
        "faiss-cpu",
        "numpy",
        "huggingface_hub",
    ]
}
report["n_chunks"] = len(chunks)
report["n_table_chunks"] = len(table_chunks)
np.save(out / "embeddings_bge-m3.npy", embeddings.astype(np.float16))
(out / "chunks_tables_1600.jsonl").write_text(
    Path("data/chunks_tables_1600.jsonl").read_text(encoding="utf-8"), encoding="utf-8"
)
(out / "embedding_report.json").write_text(json.dumps(report, indent=2))
(out / "meta.json").write_text(
    json.dumps(
        {
            "variant": "tables",
            "model": MODEL_NAME,
            "max_seq_length": MAX_LEN,
            "n": len(chunks),
            "dim": int(embeddings.shape[1]),
            "dtype": "float16",
            "normalized": True,
            "first_chunk_ids": [c["chunk_id"] for c in chunks[:3]],
        },
        indent=2,
    )
)
print(upload_folder(HF_REPO, out, "tables"))
print(json.dumps(report, indent=2))

Next: the comparison notebook (text vs table-aware retrieval on the same questions).
Copy the printed report into the chat if you want it saved in `results/`.